In [0]:
import requests, zipfile, io, os
from datetime import datetime, timezone

URL  = "https://cdn.mbta.com/MBTA_GTFS.zip"
BASE = "/Volumes/transit/bronze/landing/static"

resp = requests.get(URL, timeout=120)
resp.raise_for_status()

# Last-Modified tells you which publication this is. That's your feed version.
last_mod = resp.headers.get("Last-Modified")
feed_version = datetime.strptime(last_mod, "%a, %d %b %Y %H:%M:%S %Z").strftime("%Y-%m-%d")

out_dir = f"{BASE}/feed_version={feed_version}"
dbutils.fs.mkdirs(out_dir)

WANTED = ["stops.txt", "routes.txt", "trips.txt", "stop_times.txt",
          "calendar.txt", "calendar_dates.txt", "feed_info.txt"]

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    print("in the zip:", sorted(z.namelist()))
    for name in WANTED:
        with z.open(name) as src:
            data = src.read()
        with open(f"{out_dir}/{name}", "wb") as dst:
            dst.write(data)
        print(f"{name:24} {len(data):>12,} bytes")

print("\nfeed_version =", feed_version)

In [0]:
for name in ["stops.txt", "routes.txt", "feed_info.txt"]:
    with open(f"{out_dir}/{name}") as f:
        print(f"--- {name} ---")
        for _ in range(3):
            print(f.readline().rstrip())
        print()

In [0]:
from pyspark.sql import functions as F

# The agency's own version string — free text, kept for reference only.
feed_info = (spark.read
             .option("header", True).option("inferSchema", False)
             .option("quote", '"').option("escape", '"')
             .csv(f"{out_dir}/feed_info.txt"))
feed_info.display()

agency_feed_version = feed_info.collect()[0]["feed_version"]
print("agency version string:", repr(agency_feed_version))
print("derived feed_version: ", feed_version)

In [0]:
from pyspark.sql import functions as F

TABLES = {
    "gtfs_stops":          "stops.txt",
    "gtfs_routes":         "routes.txt",
    "gtfs_trips":          "trips.txt",
    "gtfs_stop_times":     "stop_times.txt",
    "gtfs_calendar":       "calendar.txt",
    "gtfs_calendar_dates": "calendar_dates.txt",
}

def read_gtfs(filename):
    """One GTFS file -> bronze-shaped DataFrame. All columns string."""
    return (spark.read
              .option("header", True)
              .option("inferSchema", False)
              .option("quote", '"')
              .option("escape", '"')
              .csv(f"{out_dir}/{filename}")
            .withColumn("_ingested_at",         F.current_timestamp())
            .withColumn("_source_file",         F.col("_metadata.file_path"))
            .withColumn("_feed_version",        F.lit(feed_version))
            .withColumn("_agency_feed_version", F.lit(agency_feed_version)))

counts = {}

for table, filename in TABLES.items():
    (read_gtfs(filename).write
       .format("delta")
       .mode("overwrite")
       .option("replaceWhere", f"_feed_version = '{feed_version}'")
       .saveAsTable(f"transit.bronze.{table}"))

    counts[table] = (spark.table(f"transit.bronze.{table}")
                       .filter(F.col("_feed_version") == feed_version)
                       .count())
    print(f"{table:22} {counts[table]:>12,} rows  (version {feed_version})")

In [0]:
from pyspark.sql import Row
from datetime import datetime, timezone

log_rows = [Row(feed_version=feed_version,
                agency_feed_version=agency_feed_version,
                table_name=t,
                row_count=int(c),
                ingested_at=datetime.now(timezone.utc))
            for t, c in counts.items()]

(spark.createDataFrame(log_rows)
   .write.format("delta").mode("append")
   .saveAsTable("transit.ops.ingestion_log"))

spark.table("transit.ops.ingestion_log").orderBy(F.desc("ingested_at")).display()

In [0]:
spark.table("transit.bronze.gtfs_stops").printSchema()
spark.table("transit.bronze.gtfs_stops").limit(5).display()

In [0]:
from pyspark.sql import functions as F

stops = (spark.table("transit.bronze.gtfs_stops")
           .filter(F.col("_feed_version") == feed_version))

(stops.select(
    F.count("*").alias("total"),
    F.sum((F.col("stop_desc") == "").cast("int")).alias("desc_empty_string"),
    F.sum(F.col("stop_desc").isNull().cast("int")).alias("desc_null"),
    F.sum(F.col("parent_station").isNull().cast("int")).alias("no_parent"))
 .display())

In [0]:
cols = [c for c in stops.columns if not c.startswith("_")]
(stops.select([F.sum((F.col(c) == "").cast("int")).alias(c) for c in cols])
      .display())

In [0]:
from pyspark.sql import functions as F

st = (spark.table("transit.bronze.gtfs_stop_times")
        .filter(F.col("_feed_version") == feed_version)
        .withColumn("dep_hour", F.split("departure_time", ":")[0].cast("int")))

st.filter(F.col("dep_hour") >= 24).agg(
    F.count("*").alias("over_24h"),
    F.min("departure_time").alias("earliest"),
    F.max("departure_time").alias("latest")).display()

st.select(
    F.sum((F.length("departure_time") == 7).cast("int")).alias("unpadded_times"),
    F.sum(F.col("dep_hour").isNull().cast("int")).alias("unparseable")).display()

In [0]:
RUN_BACKFILL = False   # True only to rebuild bronze from every version in landing
if not RUN_BACKFILL:
    dbutils.notebook.exit("backfill skipped")

from pyspark.sql import functions as F
from pyspark.sql import Row
from datetime import datetime, timezone

BASE = "/Volumes/transit/bronze/landing/static"

TABLES = {
    "gtfs_stops":          "stops.txt",
    "gtfs_routes":         "routes.txt",
    "gtfs_trips":          "trips.txt",
    "gtfs_stop_times":     "stop_times.txt",
    "gtfs_calendar":       "calendar.txt",
    "gtfs_calendar_dates": "calendar_dates.txt",
}

def read_csv(path):
    return (spark.read
              .option("header", True)
              .option("inferSchema", False)
              .option("quote", '"')
              .option("escape", '"')
              .csv(path))

versions = sorted(x.name.rstrip("/") for x in dbutils.fs.ls(BASE))
print("landing zone:", versions, "\n")

log_rows = []

for d in versions:
    fv     = d.split("=")[1]
    vdir   = f"{BASE}/{d}"
    agency = read_csv(f"{vdir}/feed_info.txt").collect()[0]["feed_version"]

    for table, filename in TABLES.items():
        (read_csv(f"{vdir}/{filename}")
           .withColumn("_ingested_at",         F.current_timestamp())
           .withColumn("_source_file",         F.col("_metadata.file_path"))
           .withColumn("_feed_version",        F.lit(fv))
           .withColumn("_agency_feed_version", F.lit(agency))
         .write.format("delta")
           .mode("overwrite")
           .option("replaceWhere", f"_feed_version = '{fv}'")
           .saveAsTable(f"transit.bronze.{table}"))

        n = (spark.table(f"transit.bronze.{table}")
               .filter(F.col("_feed_version") == fv).count())

        log_rows.append(Row(feed_version=fv, agency_feed_version=agency,
                            table_name=table, row_count=int(n),
                            ingested_at=datetime.now(timezone.utc)))

    print(f"{fv}  loaded   ({agency})")

(spark.createDataFrame(log_rows)
   .write.format("delta").mode("append")
   .saveAsTable("transit.ops.ingestion_log"))

print()
spark.sql("""
  SELECT _feed_version, count(*) AS stops
  FROM transit.bronze.gtfs_stops
  GROUP BY _feed_version ORDER BY _feed_version
""").display()